# Fine-tuning: Budget Query → JSON
**LoRA + 4-bit quantization (QLoRA)**  
Датасет: `budget_query_sft_train.jsonl` / `budget_query_sft_val.jsonl` из Google Drive

## 1. Установка зависимостей

In [ ]:
!pip install -q unsloth datasets trl

## 2. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Конфигурация

In [ ]:
# ─── Пути к файлам ───────────────────────────────────────────────────────────
DRIVE_ROOT      = "/content/drive/MyDrive/training/llm_sft"
TRAIN_FILE      = f"{DRIVE_ROOT}/budget_query_sft_train.jsonl"
VAL_FILE        = f"{DRIVE_ROOT}/budget_query_sft_val.jsonl"
OUTPUT_DIR      = f"{DRIVE_ROOT}/checkpoints"

# ─── Модель ──────────────────────────────────────────────────────────────────
# Можно заменить на любую другую HF-модель, например:
#   "Qwen/Qwen2-1.5B-Instruct", "mistralai/Mistral-7B-v0.1", ...
MODEL_NAME      = "Qwen/Qwen2-1.5B-Instruct"
HF_TOKEN        = None   # вставьте ваш токен если модель приватная

# ─── Обучение ────────────────────────────────────────────────────────────────
CUTOFF_LEN      = 1024
BATCH_SIZE      = 4
MICRO_BATCH     = 2
GRAD_ACCUM      = BATCH_SIZE // MICRO_BATCH
EPOCHS          = 4
LR              = 5e-5

# ─── LoRA ────────────────────────────────────────────────────────────────────
LORA_R          = 8
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05
LORA_TARGETS    = ["q_proj", "v_proj"]

print("✅ Конфиг загружен")

✅ Конфиг загружен


## 4. Загрузка и просмотр датасета

In [ ]:
import json

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_raw = load_jsonl(TRAIN_FILE)
val_raw   = load_jsonl(VAL_FILE)

print(f"Train: {len(train_raw)} примеров")
print(f"Val:   {len(val_raw)}  примеров")
print("\nПример записи:")
ex = train_raw[0]
print("  text_query:", ex.get('text_query', '')[:80])
print("  completion:", ex.get('completion', '')[:120])

Train: 426 примеров
Val:   80  примеров

Пример записи:
  text_query: Покажи объем соглашений по Свободному по месяцам
  completion: {
  "date_from": null,
  "date_to": null,
  "metrics": [
    "agreement_amount"
  ],
  "filters": {
    "source_groups":


## 5. Загрузка модели и токенайзера

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=CUTOFF_LEN,
    dtype=torch.float16,
    load_in_4bit=True,
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

model.print_trainable_parameters()
print("✅ Модель загружена")

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.4.8 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
✅ Модель загружена


## 6. LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


## 7. Токенизация датасета

In [ ]:
def tokenize_example(example):
    """
    prompt  = поле 'prompt'      (инструкция без completion)
    output  = поле 'completion'  (целевой JSON)
    labels: на части prompt ставим -100 (не считаем loss)
    """
    prompt     = example['prompt']
    completion = example['completion'].strip()
    full_text  = prompt + completion

    full_enc   = tokenizer(full_text,   truncation=True, max_length=CUTOFF_LEN, padding=False)
    prompt_enc = tokenizer(prompt,      truncation=True, max_length=CUTOFF_LEN, padding=False)

    input_ids      = full_enc['input_ids']
    attention_mask = full_enc['attention_mask']

    # добавляем EOS если влезает
    if input_ids[-1] != tokenizer.eos_token_id and len(input_ids) < CUTOFF_LEN:
        input_ids.append(tokenizer.eos_token_id)
        attention_mask.append(1)

    prompt_len = len(prompt_enc['input_ids'])
    labels = [-100] * prompt_len + input_ids[prompt_len:]

    return {
        'input_ids':      input_ids,
        'attention_mask': attention_mask,
        'labels':         labels,
    }

train_data = [tokenize_example(ex) for ex in train_raw]
val_data   = [tokenize_example(ex) for ex in val_raw]

print(f"Токенизировано — train: {len(train_data)}, val: {len(val_data)}")
print(f"Длина первого примера: {len(train_data[0]['input_ids'])} токенов")

Токенизировано — train: 426, val: 80
Длина первого примера: 878 токенов


## 8. Обучение

In [ ]:
import os
import transformers
from transformers import TrainerCallback

os.makedirs(OUTPUT_DIR, exist_ok=True)

data_collator = transformers.DataCollatorForSeq2Seq(
    tokenizer,
    pad_to_multiple_of=8,
    return_tensors='pt',
    padding=True,
    label_pad_token_id=-100,
)

class SavePeftModelCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        ckpt = os.path.join(args.output_dir, f"checkpoint-epoch-{int(state.epoch)}")
        kwargs["model"].save_pretrained(ckpt)
        print(f"\n✅ Сохранена модель после эпохи {int(state.epoch)} → {ckpt}")

training_args = transformers.TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    fp16=True,
    warmup_steps=100,
    weight_decay=0.01,
    optim="adamw_torch",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=4,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = transformers.Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    callbacks=[SavePeftModelCallback],
)

model.config.use_cache = False
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 426 | Num Epochs = 4 | Total steps = 428
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 1,089,536 of 1,544,803,840 (0.07% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.152053,0.105963
2,0.054785,0.035497
3,0.025194,0.019251
4,0.018508,0.016379



✅ Сохранена модель после эпохи 1 → /content/drive/MyDrive/training/llm_sft/checkpoints/checkpoint-epoch-1


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

...


✅ Сохранена модель после эпохи 3 → /content/drive/MyDrive/training/llm_sft/checkpoints/checkpoint-epoch-3


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


✅ Сохранена модель после эпохи 4 → /content/drive/MyDrive/training/llm_sft/checkpoints/checkpoint-epoch-4


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=428, training_loss=0.07233466235833748, metrics={'train_runtime': 1156.376, 'train_samples_per_second': 1.474, 'train_steps_per_second': 0.37, 'total_flos': 1.1947904978485248e+16, 'train_loss': 0.07233466235833748, 'epoch': 4.0})

## 9. Сохранение финальной модели

In [ ]:
trainer.evaluate()

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Финальная модель сохранена: {OUTPUT_DIR}")

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

RuntimeError: on_train_begin must be called before on_evaluate

## 10. Быстрый тест инференса

In [ ]:
model.config.use_cache = True
model.eval()

# Возьмём prompt из валидации
test_prompt = val_raw[0]['prompt']
expected    = val_raw[0]['completion']

inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("=== СГЕНЕРИРОВАНО ===")
print(generated)
print("\n=== ОЖИДАЛОСЬ ===")
print(expected)

# Проверим что это валидный JSON
try:
    parsed = json.loads(generated)
    print("\n✅ Валидный JSON")
except Exception as e:
    print(f"\n⚠️  Не валидный JSON: {e}")